In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
!ls /content/drive/MyDrive/test2/masks_test2

CNV  DRUSEN  NORMAL


In [12]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

num_classes = 3  # CNV, DRUSEN, NORMAL

model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

ckpt = torch.load("/content/oct_modelvar2_1_mobilnetv2.pth", map_location=device)

# dacă e salvat ca dict cu "state_dict"
if "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# scoate prefix "module." dacă a fost antrenat pe DataParallel
new_state = {}
for k,v in state_dict.items():
    nk = k.replace("module.", "")
    new_state[nk] = v

model.load_state_dict(new_state, strict=False)
model.to(device)
model.eval()
print("Model loaded OK")


Using: cuda
Model loaded OK


In [14]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_dir = "/content/drive/MyDrive/test2/masks_test2"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_data = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = test_data.classes
print("Classes:", class_names)


Classes: ['CNV', 'DRUSEN', 'NORMAL']


In [15]:
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        print(f"Corecte: {correct} din {total}")


        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total * 100
print(f"Acuratețe: {accuracy:.2f}%")


Corecte: 32 din 32
Corecte: 64 din 64
Corecte: 96 din 96
Corecte: 128 din 128
Corecte: 160 din 160
Corecte: 192 din 192
Corecte: 224 din 224
Corecte: 242 din 256
Corecte: 242 din 288
Corecte: 242 din 320
Corecte: 242 din 352
Corecte: 242 din 384
Corecte: 242 din 416
Corecte: 242 din 448
Corecte: 242 din 480
Corecte: 242 din 512
Corecte: 242 din 544
Corecte: 242 din 576
Corecte: 242 din 608
Corecte: 242 din 640
Corecte: 242 din 672
Corecte: 242 din 704
Corecte: 242 din 726
Acuratețe: 33.33%


In [17]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

class_names = ["CNV", "DRUSEN", "NORMAL"]

# === Classification Report ===
print("\nClassification Report :\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# === Acuratețe per clasă ===
cm = confusion_matrix(all_labels, all_preds)
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n=== Acuratețe per clasă ===")
for cls, acc in zip(class_names, class_accuracy):
    print(f"{cls}: {acc:.4f}")



Classification Report :

              precision    recall  f1-score   support

         CNV       0.33      1.00      0.50       242
      DRUSEN       0.00      0.00      0.00       242
      NORMAL       0.00      0.00      0.00       242

    accuracy                           0.33       726
   macro avg       0.11      0.33      0.17       726
weighted avg       0.11      0.33      0.17       726


=== Acuratețe per clasă ===
CNV: 1.0000
DRUSEN: 0.0000
NORMAL: 0.0000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [18]:
import torch
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

num_classes = 3  # CNV, DRUSEN, NORMAL

model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)

ckpt = torch.load("/content/oct_modelvar2_2_resnet18.pth", map_location=device)

# dacă e salvat ca dict cu "state_dict"
if "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

# scoate prefix "module." dacă a fost antrenat pe DataParallel
new_state = {}
for k,v in state_dict.items():
    nk = k.replace("module.", "")
    new_state[nk] = v

model.load_state_dict(new_state, strict=False)
model.to(device)
model.eval()
print("Model loaded OK")


Using: cuda
Model loaded OK


In [19]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

test_dir = "/content/drive/MyDrive/test2/masks_test2"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

test_data = datasets.ImageFolder(test_dir, transform=transform)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)

class_names = test_data.classes
print("Classes:", class_names)


Classes: ['CNV', 'DRUSEN', 'NORMAL']


In [20]:
correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)
        print(f"Corecte: {correct} din {total}")


        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total * 100
print(f"Acuratețe: {accuracy:.2f}%")


Corecte: 31 din 32
Corecte: 63 din 64
Corecte: 93 din 96
Corecte: 124 din 128
Corecte: 155 din 160
Corecte: 186 din 192
Corecte: 217 din 224
Corecte: 246 din 256
Corecte: 277 din 288
Corecte: 305 din 320
Corecte: 331 din 352
Corecte: 361 din 384
Corecte: 390 din 416
Corecte: 418 din 448
Corecte: 449 din 480
Corecte: 477 din 512
Corecte: 504 din 544
Corecte: 533 din 576
Corecte: 565 din 608
Corecte: 593 din 640
Corecte: 622 din 672
Corecte: 653 din 704
Corecte: 672 din 726
Acuratețe: 92.56%


In [21]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

class_names = ["CNV", "DRUSEN", "NORMAL"]

# === Classification Report ===
print("\nClassification Report :\n")
print(classification_report(all_labels, all_preds, target_names=class_names))

# === Acuratețe per clasă ===
cm = confusion_matrix(all_labels, all_preds)
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print("\n=== Acuratețe per clasă ===")
for cls, acc in zip(class_names, class_accuracy):
    print(f"{cls}: {acc:.4f}")



Classification Report :

              precision    recall  f1-score   support

         CNV       0.93      0.97      0.95       242
      DRUSEN       0.88      0.90      0.89       242
      NORMAL       0.97      0.90      0.94       242

    accuracy                           0.93       726
   macro avg       0.93      0.93      0.93       726
weighted avg       0.93      0.93      0.93       726


=== Acuratețe per clasă ===
CNV: 0.9669
DRUSEN: 0.9050
NORMAL: 0.9050
